<a href="https://colab.research.google.com/github/stefa54sm-ai/Analisi-DataSet/blob/main/Analisi_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
ANALISI DEL DATASET NATICUSdroid (Android Permissions)

SyntaxError: invalid syntax (396998750.py, line 1)

In [ ]:
# installazione delle librerie necessarie
!pip install ucimlrepo matplotlib scikit-learn seaborn numpy pandas

In [ ]:
from ucimlrepo import fetch_ucirepo

# Messaggio per confermare l'inizio dell'operazione
print("Scaricamento del dataset in corso...")

# Scarichiamo il dataset originale tramite il suo ID univoco (519)
NATICUSdroid = fetch_ucirepo(id=722)

print("Dataset scaricato con successo!")

In [ ]:
# Organizziamo i dati in tabelle chiamate DataFrame
# X contiene le caratteristiche (la presenza o meno dei permessi nelle app)
X = NATICUSdroid.data.features

# y contiene l'etichetta da prevedere (il risultato: app benigna o malware)
y = NATICUSdroid.data.targets

print("Dati estratti e pronti per l'analisi.")

In [ ]:
# Visualizziamo le informazioni generali sul dataset per capire da dove viene
print("--- Informazioni Generali (Metadata) ---")
print(NATICUSdroid.metadata)

In [ ]:
# Visualizziamo la descrizione di ogni singola colonna
print("\n--- Dettaglio delle Variabili ---")
print(NATICUSdroid.variables)

In [ ]:
# Mostriamo le prime 5 righe della tabella X per vedere i valori numerici
print("Anteprima dei dati:")
X.head()

---
INIZIAMO AD ANALIZZARE IL DATASET

Per prima cosa vogliamo verificare se il dataset che vogliamo capire è se i malware chiedono più permessi rispetto alle app normali (benigne). In poche parole mettiamo a confronto il numero totale di permessi richiesti dalle app sicure (blu) rispetto ai malware (arancione)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Uniamo le caratteristiche (X) e il risultato (y) in un'unica tabella
df = pd.concat([X, y], axis=1)

# Mappiamo i valori della colonna Result per avere etichette chiare nella legenda
# 0 = App Benigna (blu), 1 = Malware (arancione/rosso)
df['Legenda'] = df['Result'].map({0: 'App Benigna', 1: 'Malware'})

# Calcoliamo una nuova colonna: quanti permessi richiede in totale ciascuna app?
# Sommiamo i valori (1) di tutte le colonne dei permessi per ogni riga
df['total_permissions'] = X.sum(axis=1)

# Impostiamo lo stile pulito
sns.set_theme(style="whitegrid")
plt.figure(figsize=(11, 6))

# Creiamo l'istogramma suddividendo i colori in base al target ('Result': malware vs benigno)
# 'multiple="stack"' impila le barre per vedere il totale complessivo e la proporzione
sns.histplot(
    data=df,
    x='total_permissions',
    hue='Legenda',          # Colora diversamente in base al risultato (malware o no)
    bins=25,
    multiple="stack",
    palette=['#4C72B0', '#DD8452'],
    alpha=0.8
)

# Aggiungiamo titolo ed etichette
plt.title('Distribuzione del Numero Totale di Permessi per App', fontsize=16, pad=15, fontweight='bold')
plt.xlabel('Numero Totale di Permessi Richiesti', fontsize=14)
plt.ylabel('Numero di App', fontsize=14)

# Mostriamo il grafico
plt.show()


Ora, per verificare se il grafico precedente sia attendibile dobbiamo capire se il dataset che stiamo analizzando è bilanciato oppure no. Ovvero dobbiamo capire se il numero di malware e di app benigne, all'interno del dataset, è simile. Se il numero è simile allora il dataset è bilanciato e il grafico soprariportato è attendibile altrimenti dobbiamo calibrare in modo differente la nostra analisi

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Legenda',hue= 'Legenda', palette=['#4C72B0', '#DD8452'], legend = False)
plt.title('Bilanciamento del Dataset: App Benigne vs Malware', fontsize=15, pad=15, fontweight='bold')
plt.xlabel('Tipologia di App', fontsize=13)
plt.ylabel('Numero di App', fontsize=13)
plt.show()

Ora che abbiamo verificato e abbiamo confermato che il dataset sia bilanciato un altro aspetto che vogliamo analizzare è capire quali permessi sono più richiesti dai malware rispetto alle app benigne

In [ ]:
# Calcoliamo la percentuale di richiesta di ciascun permesso per malware e app benigne
perm_columns = X.columns
comparison = df.groupby('Legenda')[perm_columns].mean().T
comparison['Difference'] = abs(comparison['Malware'] - comparison['App Benigna'])

# Prendiamo i 10 permessi con la differenza maggiore (quelli più caratteristici)
top_permissions = comparison.sort_values(by='Difference', ascending=False).head(10)

# Creiamo il grafico a barre orizzontale
plt.figure(figsize=(12, 6))
top_permissions[['App Benigna', 'Malware']].plot(kind='barh', figsize=(12, 6), color=['#4C72B0', '#DD8452'])
plt.title('Top 10 Permessi più Richiesti', fontsize=15, pad=15, fontweight='bold')
plt.xlabel('Frequenza di Richiesta (Media)', fontsize=13)
plt.ylabel('Permesso Android', fontsize=13)
plt.gca().invert_yaxis() # Inverte l'asse per mettere il più importante in cima
plt.legend(title='Legenda')
plt.show()
